Investigates why real_edge facts are still misclassified as HALLUCINATION
at the best-F1 threshold (tc=0.5, tf=0.55), to determine whether this is
explainable domain-level noise (honest limitation) or something else worth
fixing before finalizing the threshold choice.

In [1]:
import pandas as pd
from pathlib import Path

INTERMEDIATE_DIR = Path("../Datasets/intermediate")
RESULTS_PATH = INTERMEDIATE_DIR / "verification_results.csv"

BEST_THETA_CONSISTENCY = 0.50
BEST_THETA_CONFIDENCE = 0.55

In [2]:
def recompute_decision(row, theta_consistency, theta_confidence):
    if row['C_f'] < theta_consistency:
        return 'HALLUCINATION', 'Level 1: low consistency'
    if row['patterns_detected'] != 'None' and pd.notna(row['patterns_detected']):
        return 'HALLUCINATION', f"Level 2: patterns ({row['patterns_detected']})"
    if row['Conf_f'] < theta_confidence:
        return 'HALLUCINATION', 'Level 3: low confidence'
    return 'VERIFIED', 'All checks passed'

In [3]:
def main():
    df = pd.read_csv(RESULTS_PATH)

    df[['decision_swept', 'reason_swept']] = df.apply(
        lambda r: pd.Series(recompute_decision(r, BEST_THETA_CONSISTENCY, BEST_THETA_CONFIDENCE)),
        axis=1
    )

    fps = df[
        (df.category == 'real_edge') &
        (df.ground_truth == 'VERIFIED') &
        (df.decision_swept == 'HALLUCINATION')
        ]

    print(f"Total real_edge false positives at tc={BEST_THETA_CONSISTENCY}, tf={BEST_THETA_CONFIDENCE}: {len(fps)}")

    print(f"\n--- Score statistics for these FPs ---")
    print(fps[['S_E', 'S_C', 'S_D', 'C_f', 'Conf_f']].describe())

    print(f"\n--- Which level triggered the FP decision? ---")
    print(fps['reason_swept'].value_counts())

    print(f"\n--- S_E distribution specifically (should be near 1.0 for TRUE real_edge facts) ---")
    print(fps['S_E'].value_counts().sort_index())

    print(f"\n--- S_D distribution (hypothesis: low/noisy S_D is driving these FPs) ---")
    print(fps['S_D'].describe())
    print(fps['S_D'].value_counts().sort_index())

    print(f"\n--- Sample of 15 FP facts, sorted by S_D ascending ---")
    sample = fps.sort_values('S_D').head(15)
    print(sample[['subject', 'predicate', 'object', 'S_E', 'S_C', 'S_D', 'C_f', 'Conf_f', 'reason_swept']].to_string(index=False))

    # Compare: for real_edge facts that were CORRECTLY verified, what does S_D look like?
    tns = df[
        (df.category == 'real_edge') &
        (df.ground_truth == 'VERIFIED') &
        (df.decision_swept == 'VERIFIED')
        ]
    print(f"\n--- For comparison: S_D distribution among CORRECTLY verified real_edge facts (n={len(tns)}) ---")
    print(tns['S_D'].describe())

In [4]:
main()

Total real_edge false positives at tc=0.5, tf=0.55: 85

--- Score statistics for these FPs ---
        S_E        S_C        S_D        C_f     Conf_f
count  85.0  85.000000  85.000000  85.000000  85.000000
mean    1.0   0.950588   0.357647   0.455943   0.723162
std     0.0   0.165920   0.193573   0.026200   0.003785
min     1.0   0.400000   0.300000   0.448140   0.722035
25%     1.0   1.000000   0.300000   0.448140   0.722035
50%     1.0   1.000000   0.300000   0.448140   0.722035
75%     1.0   1.000000   0.300000   0.448140   0.722035
max     1.0   1.000000   1.000000   0.542884   0.735721

--- Which level triggered the FP decision? ---
reason_swept
Level 1: low consistency                   78
Level 2: patterns (I3_Scale_Divergence)     7
Name: count, dtype: int64

--- S_E distribution specifically (should be near 1.0 for TRUE real_edge facts) ---
S_E
1.0    85
Name: count, dtype: int64

--- S_D distribution (hypothesis: low/noisy S_D is driving these FPs) ---
count    85.000000
mea